In [1]:
%pip install -q sentence-transformers

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
sentences = [
    "A neural network learns patterns from training data.",
    "Deep learning models learn representations from examples.",
    "The capital of France is Paris.",
    "Python is widely used for machine learning and AI development.",
    "A convolutional neural network is commonly used for image classification.",
    "Transformers use attention mechanisms to process relationships between tokens.",
    "The Eiffel Tower is a famous landmark in Paris.",
    "Semantic search retrieves text based on meaning rather than exact keywords.",
    "A database stores and organizes information for efficient retrieval.",
    "Gradient descent updates model parameters to reduce the loss."
]

embeddings = model.encode(sentences)

print("Embedding Shape: ", embeddings.shape)

Embedding Shape:  (10, 384)


In [4]:
query = "What does gradient descent do?"

query_embedding = model.encode(query)

print("Query embedding shape: ", query_embedding.shape)

Query embedding shape:  (384,)


In [5]:
from sentence_transformers import util

similarity_score = util.cos_sim(query_embedding, embeddings)

print("Similarity Score: ", similarity_score)

Similarity Score:  tensor([[0.3389, 0.3229, 0.0153, 0.1942, 0.2423, 0.2089, 0.0191, 0.1682, 0.1581,
         0.6401]])


In [6]:
scores = similarity_score[0]

top_k = 4
ranked_indices = scores.argsort(descending=True)[:top_k]

print("Query: ", query)
print("Top 4 results: -")

for idx in ranked_indices:
    i = idx.item()
    print(f"Score: {scores[i]:.4f} | {sentences[i]}")

Query:  What does gradient descent do?
Top 4 results: -
Score: 0.6401 | Gradient descent updates model parameters to reduce the loss.
Score: 0.3389 | A neural network learns patterns from training data.
Score: 0.3229 | Deep learning models learn representations from examples.
Score: 0.2423 | A convolutional neural network is commonly used for image classification.


In [7]:
documents = [
    {
        "title": "Gradient Descent",
        "text": """
        Gradient descent is an optimization algorithm used to train machine
        learning models. It updates model parameters in the direction that
        reduces the loss function. The learning rate controls the size of
        each update.
        """
    },
    {
        "title": "Neural Networks",
        "text": """
        Neural networks consist of interconnected layers of artificial
        neurons. During training, they learn patterns from examples by
        adjusting their parameters. Deep learning uses neural networks
        with multiple layers.
        """
    },
    {
        "title": "Transformers",
        "text": """
        Transformer models use attention mechanisms to represent relationships
        between tokens. They are widely used in language processing tasks,
        including text generation, translation, and semantic search.
        """
    }
]

for doc in documents:
    print(f"Title: {doc['title']}")
    print(doc["text"].strip())
    print()

Title: Gradient Descent
Gradient descent is an optimization algorithm used to train machine
        learning models. It updates model parameters in the direction that
        reduces the loss function. The learning rate controls the size of
        each update.

Title: Neural Networks
Neural networks consist of interconnected layers of artificial
        neurons. During training, they learn patterns from examples by
        adjusting their parameters. Deep learning uses neural networks
        with multiple layers.

Title: Transformers
Transformer models use attention mechanisms to represent relationships
        between tokens. They are widely used in language processing tasks,
        including text generation, translation, and semantic search.



In [8]:
from nltk.tokenize import sent_tokenize
import nltk

nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [9]:
chunks = []

for doc in documents:
    sentences_in_doc = sent_tokenize(doc["text"])

    for sentence in sentences_in_doc:
        chunks.append({
            "title": doc["title"],
            "text": sentence
        })

for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1} | {chunk['title']}")
    print(chunk["text"])
    print()

Chunk 1 | Gradient Descent

        Gradient descent is an optimization algorithm used to train machine
        learning models.

Chunk 2 | Gradient Descent
It updates model parameters in the direction that
        reduces the loss function.

Chunk 3 | Gradient Descent
The learning rate controls the size of
        each update.

Chunk 4 | Neural Networks

        Neural networks consist of interconnected layers of artificial
        neurons.

Chunk 5 | Neural Networks
During training, they learn patterns from examples by
        adjusting their parameters.

Chunk 6 | Neural Networks
Deep learning uses neural networks
        with multiple layers.

Chunk 7 | Transformers

        Transformer models use attention mechanisms to represent relationships
        between tokens.

Chunk 8 | Transformers
They are widely used in language processing tasks,
        including text generation, translation, and semantic search.



In [39]:
import re

for chunk in chunks:
  chunk["text"] = re.sub(r"\s+", " ", chunk["text"]).strip()

In [40]:
chunk_texts = [chunk["text"] for chunk in chunks]

chunk_embeddings = model.encode(chunk_texts)

print("Number of chunks:", len(chunk_texts))
print("Embedding shape:", chunk_embeddings.shape)

Number of chunks: 8
Embedding shape: (8, 384)


In [41]:
doc_query = "How does the learning rate affect training?"

doc_query_embedding = model.encode(doc_query)

print("Query embedding: ", doc_query_embedding.shape)

Query embedding:  (384,)


In [42]:
doc_similarity_score = util.cos_sim(doc_query_embedding, chunk_embeddings)

print("Similarity Score: ", doc_similarity_score)

Similarity Score:  tensor([[0.3140, 0.2809, 0.7104, 0.1504, 0.4739, 0.2869, 0.1803, 0.1224]])


In [43]:
print(doc_similarity_score)

sorted_idx = doc_similarity_score[0].argsort(descending=True)

print("Query: ", doc_query)

for idx in sorted_idx:

  print(f"Score: {doc_similarity_score[0][idx]:.4f} | {chunk_texts[idx]}")


tensor([[0.3140, 0.2809, 0.7104, 0.1504, 0.4739, 0.2869, 0.1803, 0.1224]])
Query:  How does the learning rate affect training?
Score: 0.7104 | The learning rate controls the size of each update.
Score: 0.4739 | During training, they learn patterns from examples by adjusting their parameters.
Score: 0.3140 | Gradient descent is an optimization algorithm used to train machine learning models.
Score: 0.2869 | Deep learning uses neural networks with multiple layers.
Score: 0.2809 | It updates model parameters in the direction that reduces the loss function.
Score: 0.1803 | Transformer models use attention mechanisms to represent relationships between tokens.
Score: 0.1504 | Neural networks consist of interconnected layers of artificial neurons.
Score: 0.1224 | They are widely used in language processing tasks, including text generation, translation, and semantic search.


In [44]:
def search_document(query, top_k=4):

  query_embedding = model.encode(query)
  score = util.cos_sim(query_embedding, chunk_embeddings)

  top_k_idx = score[0].argsort(descending=True)[:top_k]

  result = []

  for i in top_k_idx:

    result.append({
        "title": chunks[i]["title"],
        "text": chunks[i]["text"],
        "score": score[0][i]
    })

  return result


In [45]:
res = search_document("What are transformers?")

print("Query: What are transformers?")

for r in res:
  print(f"Title: {r['title']}")
  print(f"Text: {r['text']}")
  print(f"Score: {r['score']}")

Query: What are transformers?
Title: Transformers
Text: Transformer models use attention mechanisms to represent relationships between tokens.
Score: 0.43603596091270447
Title: Transformers
Text: They are widely used in language processing tasks, including text generation, translation, and semantic search.
Score: 0.23822206258773804
Title: Neural Networks
Text: Deep learning uses neural networks with multiple layers.
Score: 0.19796153903007507
Title: Neural Networks
Text: During training, they learn patterns from examples by adjusting their parameters.
Score: 0.15039010345935822
